# Planet Ruler Benchmark Analysis

Analysis of optimizer performance across minimizer, preset, free-parameter set, and bounds configurations.

Run from the project root after the benchmark grid completes:
```bash
jupyter notebook planet_ruler/benchmarks/visualize.ipynb
```

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

%load_ext autoreload
%autoreload 2

from planet_ruler.benchmarks.analyzer import BenchmarkAnalyzer

In [ ]:
# ── User-tunable settings ─────────────────────────────────────────────────────
DB_PATH = None  # None → default (benchmarks/results/benchmark_results.db)
ERROR_THRESHOLD = 0.10  # Reliability: fraction of runs with error < this value
CONVERGED_ONLY = False  # Exclude error/crash rows from all analyses

# Filters — set to None to include everything
IMAGE_FILTER = None  # "real" (pexels-*), "synth" (synth_*), or None for all
NOISE_SIGMA = 2  # Keep only rows with this annotation noise level (px);
# None to include all noise levels (base + all aug variants)

# Pareto weights (should sum to 1.0; set W_R_RANGE=0 to use 2-D Pareto)
W_TIME = 0.3  # Weight for total_time      (minimise)
W_ERROR = 0.5  # Weight for relative_error  (minimise)
W_R_RANGE = 0.2  # Weight for r_range_km      (maximise — wider bounds = more general)
# ─────────────────────────────────────────────────────────────────────────────
TIME_COL = "total_time"
ERROR_COL = "relative_error"
R_RANGE_COL = "r_range_configured_km"

In [ ]:
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

analyzer = BenchmarkAnalyzer(db_path=DB_PATH)
df_raw = analyzer.get_results()
df = analyzer.explode_parameters(df_raw)

if CONVERGED_ONLY:
    df = df[df["convergence_status"] == "success"].copy()

# annotation_noise_sigma column is authoritative when present (runner >= 2025-05).
# Fall back to name parsing for older DBs that lack the column.
if "annotation_noise_sigma" in df.columns:
    df["noise_sigma"] = df["annotation_noise_sigma"].fillna(0.0)
else:
    df["noise_sigma"] = (
        df["scenario_name"]
        .str.extract(r"__n([\d.]+)px_aug")[0]
        .astype(float)
        .fillna(0.0)
    )
df["is_aug"] = df["noise_sigma"] > 0

# Derived config columns used for aggregation and per-dimension analysis
if "r_lower" in df.columns and "r_upper" in df.columns:
    df[R_RANGE_COL] = ((df["r_upper"] - df["r_lower"]) / 1000).round(0)
if "r_limits_configured" in df.columns:

    def _parse_rlc(s):
        if pd.isna(s) or not s:
            return float("nan")
        lo, hi = json.loads(s)
        return round(hi - lo, 0)

    df["r_range_configured_km"] = df["r_limits_configured"].apply(_parse_rlc)
if "h_lower" in df.columns and "h_upper" in df.columns:
    df["h_range_m"] = (df["h_upper"] - df["h_lower"]).round(-2)
if "free_parameters" in df.columns:
    df["n_free"] = df["free_parameters"].apply(
        lambda x: len(x) if isinstance(x, list) else 0
    )

# h perturbation factor — parsed from perturbation_factors JSON column
if "perturbation_factors" in df.columns:
    df["h_pf"] = df["perturbation_factors"].apply(
        lambda x: float(json.loads(x).get("h", 1.0)) if pd.notna(x) and x else 1.0
    )


# Parse fit_stages JSON → sagitta-specific columns
def _parse_fit_stages(stages_json):
    try:
        stages = json.loads(stages_json) if stages_json else []
    except Exception:
        stages = []
    sag = [s for s in stages if s.get("method") == "sagitta"]
    has_opt = any(s.get("method") in ("arc", "gradient") for s in stages)
    last_sag = sag[-1] if sag else {}
    return pd.Series(
        {
            "has_sagitta": bool(sag),
            "sag_only": bool(sag) and not has_opt,
            "sag_n_sigma": last_sag.get("n_sigma"),
            "sag_bias_correct": last_sag.get("bias_correct"),
            "sag_uncertainty": last_sag.get("uncertainty"),
        }
    )


if "fit_stages" in df.columns:
    sag_cols = df["fit_stages"].apply(_parse_fit_stages)
    df = pd.concat([df, sag_cols], axis=1)
else:
    df["has_sagitta"] = False
    df["sag_only"] = False
    df["sag_n_sigma"] = None
    df["sag_bias_correct"] = None
    df["sag_uncertainty"] = None

# Actual r bounds used by the optimizer (post-sagitta intersection).
# explode_parameters already parsed parameter_limits → r_lower / r_upper.
# For sagitta rows these reflect the tightened bounds; for plain runs they equal configured limits.
if "r_lower" in df.columns and "r_upper" in df.columns:
    df["r_lo_m"] = df["r_lower"]
    df["r_hi_m"] = df["r_upper"]
    df["act_r_range_km"] = ((df["r_hi_m"] - df["r_lo_m"]) / 1000).round(1)
    df["sag_covers_true"] = (df["r_lo_m"] <= df["expected_radius"]) & (
        df["expected_radius"] <= df["r_hi_m"]
    )

print(f"Rows loaded  : {len(df_raw):,}  (after CONVERGED_ONLY filter: {len(df):,})")
print(f"  base       : {(~df['is_aug']).sum():,}")
print(f"  augmented  : {df['is_aug'].sum():,}")
if df["is_aug"].any():
    for sigma, cnt in (
        df.loc[df["is_aug"], "noise_sigma"].value_counts().sort_index().items()
    ):
        print(f"    noise={sigma:g}px : {cnt:,}")
print(f"Images       : {df['image_name'].nunique()}")
if "minimizer" in df.columns:
    print(f"Minimizers   : {sorted(df['minimizer'].dropna().unique())}")
if "minimizer_preset" in df.columns:
    print(f"Presets      : {sorted(df['minimizer_preset'].dropna().unique())}")
if "h_pf" in df.columns:
    print(f"h_pf values  : {sorted(df['h_pf'].dropna().unique())}")
if df["has_sagitta"].any():
    ws = df["has_sagitta"] & ~df["sag_only"]
    print(f"sagitta warm-start : {ws.sum():,}")
    print(f"sagitta only       : {df['sag_only'].sum():,}")
    print(
        f"  bias_correct vals: {sorted(df.loc[df['has_sagitta'], 'sag_bias_correct'].dropna().unique())}"
    )
    print(
        f"  uncertainty vals : {sorted(df.loc[df['has_sagitta'], 'sag_uncertainty'].dropna().unique())}"
    )

In [ ]:
if IMAGE_FILTER == "real":
    df = df[df["image_name"].str.startswith("pexels-")].copy()
elif IMAGE_FILTER == "synth":
    df = df[df["image_name"].str.startswith("synth_")].copy()

if NOISE_SIGMA is not None:
    df = df[df["noise_sigma"] == NOISE_SIGMA].copy()

print(
    f"After filters (IMAGE_FILTER={IMAGE_FILTER!r}, NOISE_SIGMA={NOISE_SIGMA!r}): {len(df):,} rows"
)

## Overview

In [ ]:
groups = [c for c in ["minimizer", "minimizer_preset"] if c in df.columns]
if groups:
    overview = df.groupby(groups).apply(
        lambda g: pd.Series(
            {
                "n": len(g),
                "n_base": (~g["is_aug"]).sum(),
                "n_aug": g["is_aug"].sum(),
                "mean_time": round(g[TIME_COL].mean(), 3),
                "mean_error": round(g[ERROR_COL].mean(), 4),
            }
        ),
        include_groups=False,
    )
    display(overview)

## Reliability

Fraction of runs with `relative_error < ERROR_THRESHOLD`.  
`total_time` is the only honest cross-minimizer efficiency metric — `nit` means different
things across algorithms (DE=generations, BH=configured hops, L-BFGS-B=best-restart only).

In [ ]:
def _rl(g):
    return analyzer.reliability(g, threshold=ERROR_THRESHOLD, error_col=ERROR_COL)


groups = [c for c in ["minimizer", "minimizer_preset"] if c in df.columns]

if groups:
    # Reliability on augmented rows (primary signal: robustness to annotation noise)
    for label, subset in [("aug", df[df["is_aug"]]), ("base", df[~df["is_aug"]])]:
        if subset.empty:
            continue
        rl = (
            subset.groupby(groups)
            .apply(
                lambda g: pd.Series(
                    {
                        "n": len(g),
                        "reliability": _rl(g),
                        "mean_time": round(g[TIME_COL].mean(), 2),
                        "median_time": round(g[TIME_COL].median(), 2),
                        "mean_error": round(g[ERROR_COL].mean(), 4),
                    }
                ),
                include_groups=False,
            )
            .sort_values("reliability", ascending=False)
        )
        print(f"── {label} rows ──")
        rl = rl.sort_index()
        display(rl)

In [ ]:
# ── Per-dimension reliability ─────────────────────────────────────────────────

# By r-limits width (km) — encodes how tight the radius search bounds are
if R_RANGE_COL in df.columns:
    rl_r = (
        df.groupby(R_RANGE_COL)
        .apply(
            lambda g: pd.Series(
                {
                    "n": len(g),
                    "reliability": _rl(g),
                    "mean_error": round(g[ERROR_COL].mean(), 4),
                    "mean_time": round(g[TIME_COL].mean(), 2),
                }
            ),
            include_groups=False,
        )
        .sort_index()
    )
    rl_r.index.name = "r_range_km (search width)"
    print("Reliability by r-limit width:")
    display(rl_r)

# By number of free parameters
if "n_free" in df.columns:
    rl_fp = df.groupby("n_free").apply(
        lambda g: pd.Series(
            {
                "n": len(g),
                "reliability": _rl(g),
                "mean_error": round(g[ERROR_COL].mean(), 4),
                "mean_time": round(g[TIME_COL].mean(), 2),
            }
        ),
        include_groups=False,
    )
    rl_fp.index.name = "n_free_params"
    print("\nReliability by number of free parameters:")
    display(rl_fp)

## Performance vs Accuracy

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))

color_col = "minimizer" if "minimizer" in df.columns else None
if color_col:
    palette = sns.color_palette("tab10", df[color_col].nunique())
    for i, (label, grp) in enumerate(df.groupby(color_col)):
        ax.scatter(
            grp[TIME_COL],
            grp[ERROR_COL] * 100,
            label=label,
            alpha=0.35,
            s=25,
            color=palette[i],
        )
else:
    ax.scatter(df[TIME_COL], df[ERROR_COL] * 100, alpha=0.35, s=25)

ax.axhline(
    ERROR_THRESHOLD * 100,
    color="r",
    linestyle="--",
    alpha=0.6,
    label=f"{ERROR_THRESHOLD:.0%} threshold",
)
ax.set_xlabel("Total time (s)")
ax.set_ylabel("Relative error (%)")
ax.set_title("Performance vs Accuracy by Minimizer")
ax.legend()
plt.tight_layout()
plt.show()

## Pareto Frontier

Runs are **aggregated per configuration** (median time, mean error, reliability) before computing
the frontier — so individual lucky outliers don't pollute the result.
A configuration is Pareto-optimal if no other configuration is at least as good on every metric
and strictly better on at least one.  Ranked by weighted score (lower = better).

Set `W_R_RANGE > 0` to include r-search-space width as a third dimension (maximise).

In [ ]:
# ── Aggregate to per-configuration statistics ─────────────────────────────────
config_cols = [
    c
    for c in [
        "minimizer",
        "minimizer_preset",
        "n_free",
        "r_range_configured_km",
        "h_range_m",
        "sag_n_sigma",
        "sag_bias_correct",
        "sag_uncertainty",
        "h_pf",
        "sag_only",
    ]
    if c in df.columns
]

W_TIME = 0.1  # Weight for total_time      (minimise)
W_ERROR = 0.7  # Weight for relative_error  (minimise)
W_R_RANGE = 0.2

df_agg = (
    df.groupby(config_cols, dropna=False)
    .apply(
        lambda g: pd.Series(
            {
                "n": len(g),
                TIME_COL: g[TIME_COL].median(),
                ERROR_COL: g[ERROR_COL].mean(),
                "reliability": _rl(g),
            }
        ),
        include_groups=False,
    )
    .reset_index()
)

# ── Pareto over config aggregates ─────────────────────────────────────────────
pareto_metrics = [(TIME_COL, "min"), (ERROR_COL, "min")]
score_metrics = [(TIME_COL, "min", W_TIME), (ERROR_COL, "min", W_ERROR)]

if W_R_RANGE > 0 and R_RANGE_COL in df_agg.columns:
    pareto_metrics.append((R_RANGE_COL, "max"))
    score_metrics.append((R_RANGE_COL, "max", W_R_RANGE))

pareto = analyzer.get_pareto_frontier(pareto_metrics, df=df_agg)
print(
    f"Configs total: {len(df_agg)}   Pareto-optimal: {len(pareto)}  ({len(pareto_metrics)}-D)"
)

if not pareto.empty:
    pareto = pareto.copy()
    pareto["score"] = analyzer.score_pareto(pareto, score_metrics)
    show_cols = [
        c
        for c in config_cols + [TIME_COL, ERROR_COL, "reliability", "n", "score"]
        if c in pareto.columns
    ]
    display(pareto[show_cols].sort_values("score").head(10))

In [ ]:
if not pareto.empty:
    fig, ax = plt.subplots(figsize=(10, 7))
    ax.scatter(
        df_agg[TIME_COL],
        df_agg[ERROR_COL] * 100,
        alpha=0.4,
        s=30,
        color="lightgray",
        label="all configs",
    )
    ax.scatter(
        pareto[TIME_COL],
        pareto[ERROR_COL] * 100,
        alpha=0.9,
        s=70,
        color="crimson",
        zorder=5,
        label="Pareto frontier",
    )
    ax.axhline(ERROR_THRESHOLD * 100, color="r", linestyle="--", alpha=0.4)
    ax.set_xlabel(f"Median {TIME_COL} (s)")
    ax.set_ylabel("Mean relative error (%)")
    ax.set_title("Pareto Frontier (per-configuration aggregates)")
    ax.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
# show by radius uncertainty

pareto.loc[pareto.groupby(R_RANGE_COL)["score"].idxmin()][show_cols]

In [ ]:
# also by radius uncertainty but best possible error

pareto.loc[pareto.groupby(R_RANGE_COL)["relative_error"].idxmin()][show_cols]

## Sagitta-Only Accuracy

Geometric estimate with no optimizer. Evaluation uses only the `r` point estimate.
`bias_correct` comparison reveals tilt-bias impact.

In [ ]:
if df["sag_only"].any():
    sag_only = df[df["sag_only"]].copy()

    bc_groups = [c for c in ["sag_bias_correct", "image_name"] if c in sag_only.columns]
    if bc_groups:
        so_summary = sag_only.groupby(bc_groups, dropna=False).apply(
            lambda g: pd.Series(
                {
                    "n": len(g),
                    "mean_error": round(g[ERROR_COL].mean(), 4),
                    "median_error": round(g[ERROR_COL].median(), 4),
                    "reliability": _rl(g),
                }
            ),
            include_groups=False,
        )
        print("Sagitta-only accuracy by bias_correct and image:")
        display(so_summary)

    # Error distribution: bias_correct=True vs False
    if (
        "sag_bias_correct" in sag_only.columns
        and sag_only["sag_bias_correct"].nunique() > 1
    ):
        fig, ax = plt.subplots(figsize=(8, 4))
        for bc, grp in sag_only.groupby("sag_bias_correct"):
            ax.hist(
                grp[ERROR_COL] * 100, bins=20, alpha=0.6, label=f"bias_correct={bc}"
            )
        ax.axvline(
            ERROR_THRESHOLD * 100,
            color="r",
            linestyle="--",
            alpha=0.5,
            label=f"{ERROR_THRESHOLD:.0%} threshold",
        )
        ax.set_xlabel("Relative error (%)")
        ax.set_ylabel("Count")
        ax.set_title("Sagitta-Only Error Distribution: bias_correct comparison")
        ax.legend()
        plt.tight_layout()
        plt.show()
else:
    print("No sagitta-only rows in dataset.")

## Sagitta Warm-Start Analysis

Rows that used a sagitta stage to pre-constrain `r` before the arc/gradient optimizer.
Compares `bias_correct` and `uncertainty` settings against plain arc runs.

In [ ]:
if df["has_sagitta"].any():
    sag_ws = df[df["has_sagitta"] & ~df["sag_only"]].copy()
    no_sag = df[~df["has_sagitta"]].copy()

    # ── Reliability: sagitta warm-start vs plain arc (same minimizer/preset) ──
    sag_groups = [
        c
        for c in [
            "minimizer",
            "minimizer_preset",
            "sag_n_sigma",
            "sag_bias_correct",
            "sag_uncertainty",
        ]
        if c in sag_ws.columns
    ]
    if sag_groups:
        ws_rl = (
            sag_ws.groupby(sag_groups, dropna=False)
            .apply(
                lambda g: pd.Series(
                    {
                        "n": len(g),
                        "reliability": _rl(g),
                        "mean_error": round(g[ERROR_COL].mean(), 4),
                        "mean_time": round(g[TIME_COL].mean(), 2),
                    }
                ),
                include_groups=False,
            )
            .sort_values("reliability", ascending=False)
        )
        print("Sagitta warm-start reliability by configuration:")
        display(ws_rl.head(30).sort_index())

    # ── bias_correct impact ──────────────────────────────────────────────────
    if "sag_bias_correct" in sag_ws.columns:
        bc_rl = sag_ws.groupby("sag_bias_correct", dropna=False).apply(
            lambda g: pd.Series(
                {
                    "n": len(g),
                    "reliability": _rl(g),
                    "mean_error": round(g[ERROR_COL].mean(), 4),
                    "mean_time": round(g[TIME_COL].mean(), 2),
                }
            ),
            include_groups=False,
        )
        print("\nbias_correct impact on warm-start reliability:")
        display(bc_rl.sort_index())

    # ── uncertainty method impact ────────────────────────────────────────────
    if "sag_uncertainty" in sag_ws.columns:
        unc_rl = sag_ws.groupby("sag_uncertainty", dropna=False).apply(
            lambda g: pd.Series(
                {
                    "n": len(g),
                    "reliability": _rl(g),
                    "mean_error": round(g[ERROR_COL].mean(), 4),
                    "mean_time": round(g[TIME_COL].mean(), 2),
                }
            ),
            include_groups=False,
        )
        print("\nuncertainty method impact on warm-start reliability:")
        display(unc_rl.sort_index(level="minimizer"))
else:
    print("No sagitta warm-start rows in dataset.")

## Sagitta Bound Quality

How tight are the r bounds produced by each sagitta config, and do they cover the true radius?

`act_r_range_km` is parsed from `parameter_limits["r"]` stored post-fit — for sagitta rows this
reflects the sagitta-tightened bounds after intersection with the configured limits.
`coverage_rate` is the fraction of runs where `r_lo ≤ true_r ≤ r_hi`.

A pre-stage is only useful if coverage stays high (≥95–99%) — the table and scatter below
show the tightness–coverage tradeoff by config and annotation cleanliness.

In [ ]:
if df["has_sagitta"].any() and "act_r_range_km" in df.columns:
    sag_df = df[df["has_sagitta"]].copy()

    # Bounds are deterministic given (annotation, sagitta config) — independent of
    # which minimizer follows. Deduplicate to one row per (config, image, aug variant)
    # so coverage and range stats aren't inflated by the minimizer × preset × h_limits grid.
    # Handles both old __augNN and new __nXpx_augNN naming schemes.
    sag_df["aug_id"] = (
        sag_df["scenario_name"].str.extract(r"(__(?:n[\d.]+px_)?aug\d+)$")[0].fillna("")
    )
    SAG_CFG = ["sag_n_sigma", "sag_bias_correct", "sag_uncertainty"]
    dedup_keys = SAG_CFG + ["image_name", "aug_id"]
    sag_dedup = sag_df.groupby(dedup_keys, dropna=False).first().reset_index()

    sag_dedup["cfg_r_range_km"] = sag_dedup["r_range_configured_km"]
    sag_dedup["tightening_ratio"] = (
        sag_dedup["act_r_range_km"]
        / sag_dedup["cfg_r_range_km"].replace(0, float("nan"))
    ).clip(0, 1)

    print(
        f"Unique (config × annotation) rows: {len(sag_dedup)}  "
        f"(deduplicated from {len(sag_df):,} raw sagitta rows)"
    )

    # ── Table: tightness and coverage by config × noise level ─────────────────
    agg_bnd = sag_dedup.groupby(SAG_CFG + ["noise_sigma"], dropna=False).apply(
        lambda g: pd.Series(
            {
                "n": len(g),
                "mean_range_km": round(g["act_r_range_km"].mean(), 0),
                "median_range_km": round(g["act_r_range_km"].median(), 0),
                "pct_at_cfg_limit": round(
                    (g["tightening_ratio"] >= 0.99).mean() * 100, 1
                ),
                "coverage_rate": round(g["sag_covers_true"].mean() * 100, 1),
            }
        ),
        include_groups=False,
    )
    agg_bnd.index = agg_bnd.index.set_names(SAG_CFG + ["noise_px"])
    print("\nSagitta bound quality (one row per annotation, normalized):")
    display(agg_bnd.query("sag_bias_correct == True"))

    # ── Scatter: tightness vs coverage, split by noise level ──────────────────
    noise_levels = sorted(sag_dedup["noise_sigma"].unique())
    fig, axes = plt.subplots(
        1, len(noise_levels), figsize=(7 * len(noise_levels), 5), sharey=True
    )
    if len(noise_levels) == 1:
        axes = [axes]
    n_sigma_vals = sorted(sag_dedup["sag_n_sigma"].dropna().unique())
    cmap = plt.cm.viridis
    norm = plt.Normalize(vmin=min(n_sigma_vals), vmax=max(n_sigma_vals))

    for ax, sigma in zip(axes, noise_levels):
        label = "Base (clean)" if sigma == 0 else f"noise={sigma:g}px"
        subset = sag_dedup[sag_dedup["noise_sigma"] == sigma]
        if subset.empty:
            ax.set_visible(False)
            continue
        pt = (
            subset.groupby(SAG_CFG, dropna=False)
            .apply(
                lambda g: pd.Series(
                    {
                        "mean_range_km": g["act_r_range_km"].mean(),
                        "coverage_pct": g["sag_covers_true"].mean() * 100,
                        "n": len(g),
                    }
                ),
                include_groups=False,
            )
            .reset_index()
        )
        for _, row in pt.iterrows():
            bc = bool(row["sag_bias_correct"])
            nsig = float(row["sag_n_sigma"])
            lbl = f"σ={nsig}, bc={bc}, {str(row['sag_uncertainty'])[:3]}"
            ax.scatter(
                row["mean_range_km"],
                row["coverage_pct"],
                color=cmap(norm(nsig)),
                marker="o" if bc else "x",
                s=90,
                zorder=5,
            )
            ax.annotate(
                lbl,
                (row["mean_range_km"], row["coverage_pct"]),
                fontsize=7,
                textcoords="offset points",
                xytext=(5, 3),
            )

        ax.axhline(99, ls=":", color="orange", alpha=0.7, label="99% coverage")
        ax.axhline(95, ls="--", color="red", alpha=0.5, label="95% coverage")
        ax.set_xlabel("Mean r bound width (km)\n← tighter")
        ax.set_ylabel("Coverage rate (%)" if sigma == noise_levels[0] else "")
        ax.set_title(f"{label}  (n={len(subset['aug_id'].unique())} annotations)")
        ax.legend(fontsize=8)

    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    fig.colorbar(sm, ax=axes[-1], label="n_sigma")
    fig.suptitle(
        "Sagitta bound quality: tightness vs coverage of true r\n"
        "o = bias_correct=True,  x = bias_correct=False",
        fontsize=11,
    )
    plt.tight_layout()
    plt.show()

    # ── Per-image breakdown ────────────────────────────────────────────────────
    per_img = sag_dedup.groupby(
        SAG_CFG + ["image_name", "noise_sigma"], dropna=False
    ).apply(
        lambda g: pd.Series(
            {
                "n": len(g),
                "mean_range_km": round(g["act_r_range_km"].mean(), 0),
                "coverage_pct": round(g["sag_covers_true"].mean() * 100, 1),
            }
        ),
        include_groups=False,
    )
    print("\nPer-image breakdown (deduplicated):")
    display(per_img)
else:
    print("No sagitta rows or parameter_limits not parsed — re-run setup cell.")

## Runtime Distribution

In [ ]:
if "minimizer" in df.columns and "minimizer_preset" in df.columns:
    df["config"] = df["minimizer"] + "/" + df["minimizer_preset"].fillna("?")
    fig, ax = plt.subplots(figsize=(12, 5))
    df.boxplot(column=TIME_COL, by="config", ax=ax, rot=45)
    ax.set_xlabel("minimizer / preset")
    ax.set_ylabel("Total time (s)")
    ax.set_title("Runtime Distribution by Minimizer Config")
    ax.set_yscale("log")
    plt.suptitle("")
    plt.tight_layout()
    plt.show()
elif "minimizer" in df.columns:
    fig, ax = plt.subplots(figsize=(10, 5))
    df.boxplot(column=TIME_COL, by="minimizer", ax=ax, rot=30)
    ax.set_xlabel("Minimizer")
    ax.set_ylabel("Total time (s)")
    ax.set_title("Runtime Distribution by Minimizer")
    ax.set_yscale("log")
    plt.suptitle("")
    plt.tight_layout()
    plt.show()

## Export

In [ ]:
# output_path = Path("results/benchmark_export.csv")
# analyzer.export_csv(output_path)
# print(f"Exported to: {output_path}")